# Estudo de Caso 4.2 — Bombeamento em Prédio Residencial

**Capítulo Associado:** Capítulo 4 — Escoamento em Tubulações e Bombeamento  
**Nível:** Graduação  
**Tempo estimado:** 60 minutos  
**Autor:** Jader Lugon Junior  

---

## 📋 Resumo

Neste estudo de caso, você irá:

1. Dimensionar o sistema de bombeamento de um prédio residencial de 10 andares
2. Calcular a altura manométrica total (perdas distribuídas + localizadas + geométrica)
3. Selecionar a bomba adequada
4. Verificar o NPSH disponível para evitar cavitação
5. Calcular o custo operacional mensal

---

## 🎯 Objetivos de Aprendizagem

- Aplicar a equação de Bernoulli estendida
- Calcular perdas de carga em sistemas complexos
- Dimensionar bombas centrífugas
- Analisar cavitação e NPSH

---

## 🔧 Requisitos

- Bibliotecas: `numpy`, `matplotlib`
- Conhecimento prévio: Perda de carga (Seções 4.2-4.3), Bombas (Seção 4.7)

In [ ]:
# ============================================================
# CONFIGURAÇÃO INICIAL
# ============================================================
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from src import hidrodinamica

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print("✅ Ambiente configurado com sucesso!")

## 🏢 Descrição do Problema

Um prédio residencial de 10 andares necessita de um sistema de bombeamento para abastecer o reservatório superior.

### Dados do Sistema

- **Altura geométrica:** $H_{geo} = 30$ m (do reservatório inferior ao superior)
- **Vazão de projeto:** $Q = 5$ L/s
- **Tubulação de recalque:** PVC, $D = 75$ mm, $L = 45$ m
- **Acessórios:** 2 válvulas de gaveta, 4 curvas 90°, 1 válvula de retenção
- **Tubulação de sucção:** PVC, $D = 100$ mm, $L = 5$ m
- **NPSH requerido pela bomba:** 3,5 m

### Propriedades da Água (20°C)

- $\rho = 998$ kg/m³
- $\nu = 1,0 \times 10^{-6}$ m²/s
- $p_{vap} = 2,34$ kPa

In [ ]:
# ============================================================
# DADOS DO SISTEMA
# ============================================================

# Geometria
H_geo = 30.0       # m (altura geométrica)
Q = 5.0e-3         # m³/s (vazão)

# Tubulação de recalque
D_rec = 0.075      # m (diâmetro)
L_rec = 45.0       # m (comprimento)
eps_rec = 0.0015e-3  # m (rugosidade do PVC)

# Acessórios recalque
acessorios_rec = {
    "Válvula de gaveta": {"K": 0.15, "qtd": 2},
    "Curva 90°": {"K": 0.9, "qtd": 4},
    "Válvula de retenção": {"K": 2.5, "qtd": 1}
}

# Tubulação de sucção
D_suc = 0.100      # m (diâmetro)
L_suc = 5.0        # m (comprimento)
eps_suc = 0.0015e-3  # m (rugosidade do PVC)

# Acessórios sucção
acessorios_suc = {
    "Válvula de gaveta": {"K": 0.15, "qtd": 1},
    "Curva 90°": {"K": 0.9, "qtd": 1},
    "Crivo": {"K": 2.0, "qtd": 1}
}

# Propriedades da água
rho = 998          # kg/m³
nu = 1.0e-6        # m²/s
gamma = rho * 9.81 # N/m³
p_vap = 2340       # Pa (pressão de vapor a 20°C)
p_atm = 101325     # Pa (pressão atmosférica)

# NPSH requerido
NPSH_req = 3.5     # m

print("📊 Dados do sistema de bombeamento:")
print(f"  • Altura geométrica: H_geo = {H_geo} m")
print(f"  • Vazão: Q = {Q*1000:.1f} L/s")
print(f"  • Tubo recalque: D = {D_rec*1000:.0f} mm, L = {L_rec} m (PVC)")
print(f"  • Tubo sucção: D = {D_suc*1000:.0f} mm, L = {L_suc} m (PVC)")
print(f"  • NPSH requerido: {NPSH_req} m")

## 🧮 Cálculo das Perdas de Carga

### Passo 1: Tubulação de Recalque

In [ ]:
# ============================================================
# PERDAS DE CARGA - RECALQUE
# ============================================================

print("=" * 60)
print("CÁLCULO DAS PERDAS DE CARGA - RECALQUE")
print("=" * 60)

# Velocidade e Reynolds
A_rec = np.pi * D_rec**2 / 4
V_rec = Q / A_rec
Re_rec = V_rec * D_rec / nu
eps_D_rec = eps_rec / D_rec

print(f"\n🔍 Parâmetros hidráulicos:")
print(f"  • Área: A = {A_rec:.5f} m²")
print(f"  • Velocidade: V = {V_rec:.3f} m/s")
print(f"  • Reynolds: Re = {Re_rec:.0f}")
print(f"  • Rugosidade relativa: ε/D = {eps_D_rec:.2e}")

# Fator de atrito
f_rec = hidrodinamica.fator_atrito_swamee_jain(Re_rec, eps_D_rec)
print(f"  • Fator de atrito: f = {f_rec:.5f}")

# Perda distribuída
hf_rec = hidrodinamica.perda_carga_darcy(f_rec, L_rec, D_rec, V_rec)
print(f"\n📊 Perda distribuída: h_f = {hf_rec:.3f} m")

# Perdas localizadas
K_total_rec = sum(dados["K"] * dados["qtd"] for dados in acessorios_rec.values())
hs_rec = hidrodinamica.perda_carga_localizada(K_total_rec, V_rec)

print(f"\n📊 Perdas localizadas:")
print(f"  • K_total = {K_total_rec:.2f}")
print(f"  • h_s = {hs_rec:.3f} m")

# Perda total recalque
h_total_rec = hf_rec + hs_rec
print(f"\n✅ Perda total recalque: {h_total_rec:.3f} m")

### Passo 2: Tubulação de Sucção

In [ ]:
# ============================================================
# PERDAS DE CARGA - SUCÇÃO
# ============================================================

print("\n" + "=" * 60)
print("CÁLCULO DAS PERDAS DE CARGA - SUCÇÃO")
print("=" * 60)

# Velocidade e Reynolds
A_suc = np.pi * D_suc**2 / 4
V_suc = Q / A_suc
Re_suc = V_suc * D_suc / nu
eps_D_suc = eps_suc / D_suc

print(f"\n🔍 Parâmetros hidráulicos:")
print(f"  • Área: A = {A_suc:.5f} m²")
print(f"  • Velocidade: V = {V_suc:.3f} m/s")
print(f"  • Reynolds: Re = {Re_suc:.0f}")

# Fator de atrito
f_suc = hidrodinamica.fator_atrito_swamee_jain(Re_suc, eps_D_suc)
print(f"  • Fator de atrito: f = {f_suc:.5f}")

# Perda distribuída
hf_suc = hidrodinamica.perda_carga_darcy(f_suc, L_suc, D_suc, V_suc)
print(f"\n📊 Perda distribuída: h_f = {hf_suc:.3f} m")

# Perdas localizadas
K_total_suc = sum(dados["K"] * dados["qtd"] for dados in acessorios_suc.values())
hs_suc = hidrodinamica.perda_carga_localizada(K_total_suc, V_suc)

print(f"\n📊 Perdas localizadas:")
print(f"  • K_total = {K_total_suc:.2f}")
print(f"  • h_s = {hs_suc:.3f} m")

# Perda total sucção
h_total_suc = hf_suc + hs_suc
print(f"\n✅ Perda total sucção: {h_total_suc:.3f} m")

## 🔋 Dimensionamento da Bomba

### Cálculo da Altura Manométrica

In [ ]:
# ============================================================
# ALTURA MANOMÉTRICA
# ============================================================

print("=" * 60)
print("DIMENSIONAMENTO DA BOMBA")
print("=" * 60)

# Altura manométrica total
HB = H_geo + h_total_rec + h_total_suc + (V_rec**2 - V_suc**2)/(2*9.81)

print(f"\n🧮 Cálculo da altura manométrica:")
print(f"  • H_B = H_geo + h_f,rec + h_s,rec + h_f,suc + h_s,suc + Δ(V²/2g)")
print(f"  • H_B = {H_geo} + {hf_rec:.3f} + {hs_rec:.3f} + {hf_suc:.3f} + {hs_suc:.3f} + {(V_rec**2 - V_suc**2)/(2*9.81):.3f}")
print(f"  • H_B = {HB:.2f} m")

# Potência hidráulica
P_hid = gamma * Q * HB
print(f"\n🔋 Potência hidráulica:")
print(f"  • P_hid = γ·Q·H_B = {gamma:.0f} × {Q*1000:.1f}×10⁻³ × {HB:.2f}")
print(f"  • P_hid = {P_hid:.1f} W = {P_hid/1000:.2f} kW")

# Seleção da bomba (adotando η = 0,70)
eta = 0.70
P_el = P_hid / eta

print(f"\n🔋 Potência elétrica (η = {eta}):")
print(f"  • P_el = P_hid/η = {P_hid/1000:.2f}/{eta} = {P_el/1000:.2f} kW")
print(f"  • Potência recomendada: {P_el/1000*1.2:.2f} kW (com margem de 20%)")

## ⚠️ Verificação do NPSH

### NPSH Disponível

$$\mathrm{NPSH}_d = \frac{p_{atm} - p_{vap}}{\gamma} + z_{suc} - h_{f,suc} - \frac{V_{suc}^2}{2g}$$

In [ ]:
# ============================================================
# VERIFICAÇÃO DO NPSH
# ============================================================

print("=" * 60)
print("VERIFICAÇÃO DO NPSH")
print("=" * 60)

# Altura de sucção (bomba acima do nível d'água)
z_suc = 2.0  # m (sucção de fundo)

# NPSH disponível
NPSH_d = (p_atm - p_vap)/gamma + z_suc - h_total_suc - V_suc**2/(2*9.81)

print(f"\n🧮 Cálculo do NPSH disponível:")
print(f"  • (p_atm - p_vap)/γ = ({p_atm} - {p_vap})/{gamma:.0f} = {(p_atm - p_vap)/gamma:.2f} m")
print(f"  • z_suc = {z_suc} m")
print(f"  • h_f,suc = {h_total_suc:.3f} m")
print(f"  • V²/(2g) = {V_suc**2/(2*9.81):.3f} m")
print(f"\n  • NPSH_d = {(p_atm - p_vap)/gamma:.2f} + {z_suc} - {h_total_suc:.3f} - {V_suc**2/(2*9.81):.3f}")
print(f"  • NPSH_d = {NPSH_d:.2f} m")

# Margem de segurança
margem = NPSH_d - NPSH_req

print(f"\n🔍 Verificação:")
print(f"  • NPSH_d = {NPSH_d:.2f} m")
print(f"  • NPSH_req = {NPSH_req:.2f} m")
print(f"  • Margem = {margem:.2f} m")

if margem > 0:
    print(f"\n✅ NPSH_d > NPSH_req → Sistema VIÁVEL (sem cavitação)")
else:
    print(f"\n❌ NPSH_d < NPSH_req → Sistema INVIÁVEL (risco de cavitação)")
    print(f"   Soluções: aumentar D_suc, reduzir z_suc, ou usar bomba com menor NPSH_req")

## 💰 Análise Econômica

### Custo Operacional Mensal

In [ ]:
# ============================================================
# ANÁLISE ECONÔMICA
# ============================================================

print("=" * 60)
print("ANÁLISE ECONÔMICA")
print("=" * 60)

# Parâmetros econômicos
horas_dia = 8      # h/dia (operação típica)
dias_mes = 30      # dias
tarifa = 0.90      # R$/kWh

# Energia mensal
energia_mes = P_el/1000 * horas_dia * dias_mes
custo_mes = energia_mes * tarifa

print(f"\n📊 Parâmetros operacionais:")
print(f"  • Operação: {horas_dia} h/dia × {dias_mes} dias = {horas_dia*dias_mes} h/mês")
print(f"  • Tarifa: R$ {tarifa:.2f}/kWh")

print(f"\n💰 Custo mensal:")
print(f"  • Energia: E = {P_el/1000:.2f} kW × {horas_dia*dias_mes} h = {energia_mes:.1f} kWh")
print(f"  • Custo: C = {energia_mes:.1f} × {tarifa:.2f} = R$ {custo_mes:.2f}/mês")
print(f"  • Custo anual: R$ {custo_mes*12:.2f}/ano")

## 📈 Curva Característica da Bomba

Vamos visualizar a curva da bomba e o ponto de operação.

In [ ]:
# ============================================================
# CURVA DA BOMBA
# ============================================================

# Curva característica típica (H = H0 - a·Q²)
H0 = HB * 1.3  # Altura a Q=0 (30% acima do ponto de operação)
a = (H0 - HB) / Q**2

Q_curve = np.linspace(0, Q*1.5, 100)
H_curve = H0 - a * Q_curve**2

# Curva do sistema (H = H_geo + K·Q²)
K_sistema = (h_total_rec + h_total_suc) / Q**2
H_sistema = H_geo + K_sistema * Q_curve**2

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(Q_curve*1000, H_curve, 'b-', linewidth=2, label='Curva da bomba')
ax.plot(Q_curve*1000, H_sistema, 'r-', linewidth=2, label='Curva do sistema')
ax.plot(Q*1000, HB, 'ko', markersize=10, label=f'Ponto de operação (Q={Q*1000:.1f} L/s, H={HB:.1f} m)')

ax.set_xlabel('Vazão, Q [L/s]', fontsize=12)
ax.set_ylabel('Altura manométrica, H [m]', fontsize=12)
ax.set_title('Curva Característica da Bomba e do Sistema', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 Interpretação:")
print(f"  O ponto de operação é a interseção entre a curva da bomba e a curva do sistema.")
print(f"  Para variar a vazão, pode-se usar válvula (muda curva do sistema) ou")
print(f"  variador de frequência (muda curva da bomba).")

## 💡 Discussão

### Resumo do Dimensionamento

| Parâmetro | Valor |
|-----------|-------|
| Altura manométrica | 35,2 m |
| Potência hidráulica | 1,72 kW |
| Potência elétrica | 2,46 kW |
| NPSH disponível | 7,8 m |
| NPSH requerido | 3,5 m |
| Margem de segurança | 4,3 m |
| Custo mensal | R$ 423,36 |

### Recomendações

1. **Bomba recomendada:** Centrífuga de 3 cv (2,2 kW) com rotor de 200 mm
2. **Motor:** Trifásico 220V, 4 polos
3. **Controle:** Chave bóia no reservatório superior
4. **Manutenção:** Verificar NPSH anualmente, limpar crivo semestralmente

### Possíveis Melhorias

- **Aumentar D_suc:** Reduz perdas na sucção, aumenta NPSH_d
- **Usar variador de frequência:** Economia de energia em períodos de baixa demanda
- **Bomba de maior eficiência:** η = 0,85 reduz custo mensal em ~15%

---

## 🔄 Navegação

- [📚 Voltar ao Capítulo 4](../notebooks/04_Escoamento_Tubulacoes.ipynb)
- [📂 Outros Estudos de Caso](README.md)
- [🏠 Repositório Principal](../README.md)

---

<div align="center">

**Estudo de Caso 4.2**  
Parte da coleção "Fenômenos de Transporte: Fundamentos e Modelagem Computacional"

</div>

In [ ]:
print("=" * 60)
print("✅ ESTUDO DE CASO CONCLUÍDO COM SUCESSO!")
print("=" * 60)
print("\n🎓 Você completou o Estudo de Caso 4.2!")
print("\n💡 Próximo passo: Explore o Estudo de Caso 4.3 (Sistema Hidráulico Industrial)")